<a href="https://colab.research.google.com/github/dalmasm/AI/blob/Data-Science/ejercicio_llm_kaggle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 9.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is i

In [ ]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [ ]:
# 1️⃣ Cargar el dataset
train_df = pd.read_csv("/content/train.csv")
test_df = pd.read_csv("/content/test.csv")

print("CUDA disponible:", torch.cuda.is_available())  # ¿Devuelve True o False?
print("Número de GPUs:", torch.cuda.device_count())  # ¿Cuántas GPUs detecta?
if torch.cuda.is_available():
    print("GPU en uso:", torch.cuda.current_device())
    print("Nombre de la GPU:", torch.cuda.get_device_name(0))



CUDA disponible: True
Número de GPUs: 1
GPU en uso: 0
Nombre de la GPU: Tesla T4


In [ ]:
# 2️⃣ Preprocesamiento: Codificar etiquetas
label_map = {"a": 0, "b": 1, "tie": 2}
train_df["winner_model"] = train_df[["winner_model_a", "winner_model_b", "winner_tie"]].idxmax(axis=1).map(lambda x: x.split("_")[-1])
train_df["winner_model"] = train_df["winner_model"].map(label_map)

# 3️⃣ Tokenización
model_name = "albert-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)


def tokenize_function(examples):
    return tokenizer(
        examples["prompt"],
        [a + " [SEP] " + b for a, b in zip(examples["response_a"], examples["response_b"])],
        truncation=True, padding="max_length", max_length=128
    )

train_dataset = Dataset.from_pandas(train_df[["prompt", "response_a", "response_b", "winner_model"]])
train_dataset = train_dataset.map(tokenize_function, batched=True)
train_dataset = train_dataset.remove_columns(["prompt", "response_a", "response_b"])
train_dataset = train_dataset.rename_column("winner_model", "labels")
train_dataset = train_dataset.select(range(10000))  # Usa solo 20K filas en lugar de 55K
train_dataset.set_format("torch")



# 4️⃣ Dividir en entrenamiento y validación
train_test_split = train_dataset.train_test_split(test_size=0.1)
train_dataset = train_test_split["train"]
eval_dataset = train_test_split["test"]


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/760k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

Map:   0%|          | 0/57477 [00:00<?, ? examples/s]

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
# 5️⃣ Cargar modelo y configurar entrenamiento
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)  # Mueve manualmente el modelo a la GPU o CPU


training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    fp16=True,  # Mantener entrenamiento en media precisión
    gradient_accumulation_steps=2,
    dataloader_num_workers=8,
    remove_unused_columns=False,  # Solución al error
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset
)

# 6️⃣ Entrenar el modelo
trainer.train()



Some weights of AlbertForSequenceClassification were not initialized from the model checkpoint at albert-base-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Step,Training Loss
50,1.116300
100,1.112900
150,1.109800
200,1.109700
250,1.100300
300,1.093400
350,1.079900
400,1.083900
450,1.089200
500,1.093900


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


✅ ¡Código completado! Archivo 'submission.csv' generado.


In [ ]:
# 7️⃣ Generar predicciones en el test set
test_dataset = Dataset.from_pandas(test_df)
test_dataset = test_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.remove_columns(["id"])
test_dataset.set_format("torch")

predictions = trainer.predict(test_dataset).predictions
pred_labels = torch.argmax(torch.tensor(predictions), dim=1).numpy()
submission_df = pd.DataFrame({"id": test_df["id"], "winner_model": [list(label_map.keys())[i] for i in pred_labels]})

# 8️⃣ Guardar resultados
submission_df.to_csv("submission.csv", index=False)

print("✅ ¡Código completado! Archivo 'submission.csv' generado.")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

# Obtener predicciones en el dataset de validación
results = trainer.predict(eval_dataset)
preds = torch.argmax(torch.tensor(results.predictions), dim=1).numpy()
labels = eval_dataset["labels"]

# Calcular métricas
accuracy = accuracy_score(labels, preds)
f1 = f1_score(labels, preds, average="weighted")  # Usa "macro" si tienes clases equilibradas

print(f"✅ Precisión (Accuracy): {accuracy:.4f}")
print(f"✅ F1-score (Weighted): {f1:.4f}")


✅ Precisión (Accuracy): 0.3930
✅ F1-score (Weighted): 0.3893
